In [2]:
!python --version

Python 3.11.14


In [3]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv/perturb_agent
SRC added: /home/flavio/uv/perturb_agent/src


### Defaults

In [4]:
ROOT = Path().resolve().parent
root0 = ROOT / "data"

gdc = GDC(root0=root0)

os.listdir(root0)[:10]


['cancer',
 'reactome_summary.tsv',
 'reactome',
 'vector_store',
 'reactome_pathways.tsv',
 'TCGA',
 'gdc_programs.txt']

### Get all programs

In [5]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/gdc_programs.txt'


In [6]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [7]:
gdc.url_gdc_project

'https://api.self.cancer.gov/projects'

In [8]:
prog_id = 'TCGA'
df_psi = gdc.get_primary_sites(prog_id=prog_id, force=force, verbose=verbose)

psi_id_list = np.unique(df_psi.psi_id)
psi_id_list

Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/TCGA/primary_site_program_TCGA.tsv'


array(['TCGA-ACC', 'TCGA-BLCA', 'TCGA-BRCA', 'TCGA-CESC', 'TCGA-CHOL',
       'TCGA-COAD', 'TCGA-DLBC', 'TCGA-ESCA', 'TCGA-GBM', 'TCGA-HNSC',
       'TCGA-KICH', 'TCGA-KIRC', 'TCGA-KIRP', 'TCGA-LAML', 'TCGA-LGG',
       'TCGA-LIHC', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-MESO', 'TCGA-OV',
       'TCGA-PAAD', 'TCGA-PCPG', 'TCGA-PRAD', 'TCGA-READ', 'TCGA-SARC',
       'TCGA-SKCM', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM',
       'TCGA-UCEC', 'TCGA-UCS', 'TCGA-UVM'], dtype=object)

In [9]:
psi_id = 'TCGA-BRCA'

gdc.set_primary_site(psi_id = psi_id)

primary_site = gdc.primary_site
psi_id = gdc.psi_id

psi_id, primary_site

('TCGA-BRCA', 'Breast')

In [11]:
verbose=False

df_cases, df_tumor_samples, df_tummor_mut, tumor_barcode_list = gdc.get_filtered_tables(psi_id=psi_id, sample_type_term='Primary Tumor', verbose=verbose)
len(df_cases)

255

In [12]:
df_cases.head(2)

,primary_site,disease_type,case_id,diagnoses,psi_id,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,primary_site_norm,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac
0,Breast,Ductal and Lobular Neoplasms,c25898b4-eb33-42f4-bf3f-bf532a929e7d,"[{'primary_diagnosis': 'Lobular carcinoma, NOS...",TCGA-BRCA,lobular,unknown,"Lobular carcinoma, NOS",NaN,NaN,...,breast,ductal and lobular neoplasms,lobular carcinoma,other,epithelial,breast_lobular,ok,valid,1,0.000911
1,Breast,Ductal and Lobular Neoplasms,bb8d42d3-ad65-4d88-ae1d-f9aadfc7962d,"[{'primary_diagnosis': 'Lobular carcinoma, NOS...",TCGA-BRCA,lobular,unknown,"Lobular carcinoma, NOS",NaN,NaN,...,breast,ductal and lobular neoplasms,lobular carcinoma,other,epithelial,breast_lobular,ok,valid,1,0.000911


In [13]:
case_id_list = np.unique(df_cases.case_id)
print(len(case_id_list))

case_id_list[:3]

255


array(['00807dae-9f4a-4fd1-aac2-82eb11bf2afb',
       '00b11ca8-8540-4a3d-b602-ec754b00230b',
       '02bbb632-0f7f-439d-b8f0-c86a06237424'], dtype=object)

In [14]:
row = df_cases.iloc[0]

subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue

subtype_global, tumor_class, subtype_global

('lobular', 'other', 'lobular')

In [ ]:
df_cases.head(3)

### Preparing to get VCF files from TCGA programa

In [ ]:
case_id_list = np.unique(df_cases.case_id)
print(len(case_id_list))

row = df_cases.iloc[0]

subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue

print(subtype_global, tumor_class, subtype_global)

In [15]:
verbose=True
force=False

df_vcf = gdc.get_VCF_files(subtype_global=subtype_global, tumor_class=tumor_class, subtype_tissue=subtype_tissue, 
                       batch_cases=20, batch_size=20, timeout=120, force=force, verbose=verbose)

print(len(df_vcf))
                    

Table opened ((1098, 23)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/cases_for_TCGA-BRCA.tsv'
>>> 1098 cases
Table opened ((354, 14)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/vcf_files_for_TCGA-BRCA_Breast_subtype_lobular_tumor_other_subtype_tissue_breast_lobular.tsv'
354


In [16]:
df_vcf.head(3).T

,0,1,2
case_id,0c1023c0-d2ce-4940-8047-7642f1dc3965,0c1023c0-d2ce-4940-8047-7642f1dc3965,0c1023c0-d2ce-4940-8047-7642f1dc3965
submitter_id,TCGA-BH-A0HB,TCGA-BH-A0HB,TCGA-BH-A0HB
sample_id,19193b8f-1e1b-448c-b934-93f25e60d45b,19193b8f-1e1b-448c-b934-93f25e60d45b,19193b8f-1e1b-448c-b934-93f25e60d45b
sample_type,Primary Tumor,Primary Tumor,Primary Tumor
barcode_sample,TCGA-BH-A0HB-01A,TCGA-BH-A0HB-01A,TCGA-BH-A0HB-01A
file_id,a27f1856-e4b3-4e0c-a285-0a21ff68219c,41253061-634e-4b6c-933c-87f27f88fb35,c51a4a1d-3931-4ae5-be01-db9a4c92af9c
file_name,71e0809f-c79f-4f48-935b-14b0b58d60e9.svaba.som...,TCGA-BRCA.f4c82250-7bba-4a41-87a0-93f97573baab...,06857ea5-33b5-45b8-9795-a8f68197af2a.svaba.som...
data_type,Raw Simple Somatic Mutation,Raw Simple Somatic Mutation,Raw Simple Somatic Mutation
data_format,VCF,VCF,VCF
workflow_type,SvABA Indel,GATK4 MuTect2,SvABA Indel


In [17]:
np.unique(df_vcf.data_type)

array(['Raw Simple Somatic Mutation'], dtype=object)

In [18]:
verbose=True
force=False

row = df_vcf.iloc[0]

data_type = row.data_type
case_id = row.case_id
file_id = row.file_id


data_type, case_id, file_id

('Raw Simple Somatic Mutation',
 '0c1023c0-d2ce-4940-8047-7642f1dc3965',
 'a27f1856-e4b3-4e0c-a285-0a21ff68219c')

In [19]:
filename = gdc.get_table_given_fileID(case_id=case_id, file_id=file_id, sample_type='Tumor', file_type=data_type, force=force, verbose=verbose)
filename

Downloading: Error: 403
URL: https://api.gdc.cancer.gov/data/a27f1856-e4b3-4e0c-a285-0a21ff68219c
Content-Type: application/json
Preview: {"message":"Cannot not allow read access to file(s): a27f1856-e4b3-4e0c-a285-0a21ff68219c."}



###  'access': 'controlled'

In [20]:
import requests

file_id = "a27f1856-e4b3-4e0c-a285-0a21ff68219c"

url = f"https://api.gdc.cancer.gov/files/{file_id}"
params = {
    "fields": "file_id,file_name,data_type,data_format,access,state"
}

r = requests.get(url, params=params)
print(r.json())

{'data': {'data_format': 'VCF', 'access': 'controlled', 'file_name': '71e0809f-c79f-4f48-935b-14b0b58d60e9.svaba.somatic.indel.vcf.gz', 'file_id': 'a27f1856-e4b3-4e0c-a285-0a21ff68219c', 'data_type': 'Raw Simple Somatic Mutation', 'state': 'released'}, 'warnings': {}}
